# 01c  -  Health Insurance Claims (provided labels)

Replacement for `01_health_insurance_labeling.ipynb` when you have the **NHIS claims workbook with fraud labels**.
The tariff-rule labelling is not needed: `fraud_label` comes with the data.

**Input:** `data/NHIS_claims_deidentified.csv` (create it with `python tools/deidentify_nhis.py <workbook.xlsx>`)  
**Outputs:** `outputs/health_insurance_labeled.csv` (model table, same path the later notebooks read) and
`outputs/health_insurance_groups.csv` (`row_id, member_id, admission_date, fraud_label` for grouped/temporal splits)

**Why a restricted feature table?** The later notebooks target-encode any text column with more than 10 values, which leaks the label.
So this notebook outputs only numeric columns and low-cardinality categories (at most 9 levels), and **no identifiers or free text**.
`member_id` is written only to the groups file, to be used for `GroupKFold`, never as a feature.

In [ ]:
exec(open('00_config.py').read())
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load and validate

In [ ]:
if not os.path.exists(HI_CLAIMS_CSV):
    raise FileNotFoundError(f"{HI_CLAIMS_CSV} not found. Run: python tools/deidentify_nhis.py <NHIS_CLAIM_DATASET.xlsx>")

claims = pd.read_csv(HI_CLAIMS_CSV)
required = ['row_id','member_id','age_at_admission','gender','admission_date','discharge_date','length_of_stay',
            'procedures','icd10','diagnoses','gdrg','services_ghc','pharmacy_ghc','total_ghc','fraud_label']
missing = [c for c in required if c not in claims.columns]
assert not missing, f'Missing columns: {missing}'

# Safety: refuse to continue if direct identifiers are present
PII = {'surname','other names','dob','member no.','folder no.','ccc','name','names'}
leaked = [c for c in claims.columns if c.strip().lower() in PII]
assert not leaked, f'Direct identifiers present: {leaked}. Use the de-identified file.'

assert set(claims['fraud_label'].unique()) <= {0, 1}, 'fraud_label must be 0/1 with 1 = fraud'
print(f'Claims: {claims.shape}   members: {claims.member_id.nunique():,}')
print(f'Fraud : {int(claims.fraud_label.sum())} of {len(claims):,}  ({claims.fraud_label.mean():.2%})')

## 2. Label and data sanity checks

In [ ]:
n_pos = int(claims.fraud_label.sum())
if n_pos < 200:
    print(f'WARNING: only {n_pos} positive claims. A single 15% test split holds about {int(n_pos*0.15)} frauds, '
          f'so use repeated grouped cross-validation and report intervals.')

amt = claims[['services_ghc','pharmacy_ghc','total_ghc']]
print('Missing amounts      :', int(amt.isna().sum().sum()))
print('Total != Services+Pharmacy :', int(((claims.total_ghc - claims.services_ghc.fillna(0) - claims.pharmacy_ghc.fillna(0)).abs() > 1).sum()))
print('Negative length of stay    :', int((claims.length_of_stay < 0).sum()))
print('Duplicate rows (ex row_id) :', int(claims.drop(columns=['row_id']).duplicated().sum()))
print()
print('Mean total by label (GHS):')
print(claims.groupby('fraud_label')['total_ghc'].agg(['count','mean','median']).round(2).to_string())

# Members with claims of both classes: why splits must be grouped by member
both = (claims.groupby('member_id')['fraud_label'].nunique() > 1).sum()
multi = (claims.groupby('member_id').size() > 1).sum()
print(f'\nMembers with >1 claim: {multi:,}   members with both labels: {both}')

# Shortcut check: is any single missing-value pattern (almost) the label?
print('\nShortcut check: missing ICD-10 code vs label')
ct = pd.crosstab(claims['icd10'].isna().rename('icd10_missing'), claims['fraud_label'].rename('fraud_label'))
print(ct.to_string())
miss_fraud = int(((claims['icd10'].isna()) & (claims['fraud_label'] == 1)).sum())
miss_legit = int(((claims['icd10'].isna()) & (claims['fraud_label'] == 0)).sum())
if miss_fraud and not miss_legit:
    print(f'WARNING: ICD-10 is missing on {miss_fraud} fraudulent claims and on {miss_legit} legitimate ones '
          f'({miss_fraud/int(claims.fraud_label.sum()):.0%} of all fraud). The flag is excluded from the features unless '
          f'HI_USE_MISSING_ICD_FLAG = True in 00_config.py. Report results both ways.')
print('Pharmacy bill is zero: fraud {:.0%}  legitimate {:.0%}'.format(
    (claims.loc[claims.fraud_label == 1, 'pharmacy_ghc'] == 0).mean(), (claims.loc[claims.fraud_label == 0, 'pharmacy_ghc'] == 0).mean()))

## 3. Build the model table (no identifiers, no free text, no label-derived columns)

Categories are collapsed to at most 9 levels using **label-free** frequency counts, so the later one-hot step is used and target encoding is avoided.

In [ ]:
adm = pd.to_datetime(claims['admission_date'], errors='coerce')

def top_levels(s, n=8, other='OTHER'):
    keep = s.value_counts().head(n).index
    return s.where(s.isin(keep), other).fillna(other)

gdrg = claims['gdrg'].astype(str).str.strip().str.upper()
icd  = claims['icd10'].astype(str).str.strip().str.upper().replace({'NAN': np.nan})

feat = pd.DataFrame({
    'age_at_admission': claims['age_at_admission'],
    'gender':           claims['gender'].fillna('Unknown'),
    'length_of_stay':   claims['length_of_stay'],
    'admission_month':  adm.dt.month,
    'admission_dow':    adm.dt.dayofweek,
    'services_ghc':     claims['services_ghc'],
    'pharmacy_ghc':     claims['pharmacy_ghc'],
    'total_ghc':        claims['total_ghc'],
    'pharmacy_share':   (claims['pharmacy_ghc'] / claims['total_ghc'].replace(0, np.nan)).clip(0, 1),
    'has_procedure':    claims['procedures'].notna().astype(int),
    'gdrg_group':       top_levels(gdrg.str[:4]),                       # specialty prefix, e.g. OPDC, MEDI
    'gdrg_frequency':   gdrg.map(gdrg.value_counts(normalize=True)),     # label-free share of claims with this G-DRG code
    'gdrg_age_suffix':  gdrg.str[-1].where(gdrg.str[-1].isin(['A', 'C']), 'OTHER'),
    'icd10_chapter':    top_levels(icd.str[:1]),                         # first letter of ICD-10
})
if HI_USE_MISSING_ICD_FLAG:
    feat['has_icd10'] = icd.notna().astype(int)      # shortcut feature, see section 2
feat[TARGET_COL] = claims['fraud_label'].astype(int)

# Guards that protect the later notebooks
obj = feat.drop(columns=[TARGET_COL]).select_dtypes(exclude=np.number)
too_rich = [c for c in obj.columns if obj[c].nunique() > 10]
assert not too_rich, f'High-cardinality text columns would be target-encoded downstream: {too_rich}'
assert not any(c in feat.columns for c in ['member_id','folder_id','row_id']), 'identifier leaked into features'
print(f'Model table: {feat.shape}   categorical: {obj.columns.tolist()}')
miss = feat.isna().sum(); miss = miss[miss > 0]
print(miss.to_string() if len(miss) else 'No missing values')

## 4. Save

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
feat.to_csv(HI_LABELED_CSV, index=False)

groups = pd.DataFrame({'row_id': claims['row_id'], 'member_id': claims['member_id'],
                       'admission_date': claims['admission_date'], TARGET_COL: claims['fraud_label']})
groups.to_csv(HI_GROUPS_CSV, index=False)

print('Saved model table ->', os.path.relpath(HI_LABELED_CSV, BASE_DIR), feat.shape)
print('Saved groups      ->', os.path.relpath(HI_GROUPS_CSV, BASE_DIR), groups.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
cnt = feat[TARGET_COL].value_counts().sort_index()
axes[0].bar(['Legitimate', 'Fraud'], [cnt.get(0, 0), cnt.get(1, 0)], color=['#E65100', '#C62828'])
axes[0].set_title(f'Class distribution (fraud {feat[TARGET_COL].mean():.2%})')
for i, v in enumerate([cnt.get(0, 0), cnt.get(1, 0)]):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontweight='bold')
for lab, color in [(0, '#E65100'), (1, '#C62828')]:
    axes[1].hist(np.log10(feat.loc[feat[TARGET_COL] == lab, 'total_ghc'].clip(lower=1)), bins=40,
                 alpha=0.6, density=True, color=color, label=['Legitimate', 'Fraud'][lab])
axes[1].set_title('Claim total (log10 GHS)'); axes[1].legend()
rate = feat.groupby('gdrg_group')[TARGET_COL].agg(['mean', 'count']).sort_values('mean')
axes[2].barh(rate.index, rate['mean'] * 100, color='#6A1B9A', alpha=0.85)
axes[2].set_title('Fraud rate by G-DRG group (%)')
plt.suptitle('NHIS claims (provided labels)', fontweight='bold')
plt.tight_layout()
os.makedirs(FIGURE_DIR, exist_ok=True)
plt.savefig(os.path.join(FIGURE_DIR, '01c_health_insurance_claims.png'), dpi=150, bbox_inches='tight')
plt.show()

## Summary and cautions

| Item | Detail |
|---|---|
| Labels | provided by the NHIS workbook; `fraud_label = 1` is fraud (the source column is inverted) |
| Features | demographics, amounts, dates, collapsed G-DRG and ICD-10 groups; no identifiers or free text |
| Positives | few (under 1% of claims): use grouped, repeated CV and report intervals; prefer PR-AUC or recall at a review budget |
| Splitting | use `outputs/health_insurance_groups.csv` with `GroupKFold` on `member_id`; consider a temporal hold-out on the last months |
| Shortcut | ICD-10 is missing on 60 of 87 frauds and on no legitimate claim; `has_icd10` is excluded by default (`HI_USE_MISSING_ICD_FLAG`); report both |
| Concentration | most fraud sits in a few outpatient G-DRG codes; also evaluate within them |

-> **Next: `02_preprocessing.ipynb`** (set `HI_LABEL_SOURCE = "provided"` in `00_config.py`).